# Stack Overflow Agentic RAG

An LLM searches Stack Overflow, selects relevant questions, reads their answers, and writes a response with source links.

**Before running:** use `uv sync --locked --extra notebook`, set `OLLAMA_API_KEY` in `.env`, and launch with `uv run --extra notebook jupyter lab notebooks/demo.ipynb`. Select the project’s `.venv` kernel.


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# Locate .env when Jupyter starts at either the repository root or notebooks/.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/stackoverflow_agentic_rag").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from inside the Stack Overflow Agentic RAG project.")
load_dotenv(project_root / ".env")

True

In [3]:
from stackoverflow_agentic_rag import DEFAULT_MODEL, AgenticRAG

api_key = os.getenv("LLM_API_KEY") or os.getenv("OLLAMA_API_KEY")
if not api_key:
    raise ValueError("Set OLLAMA_API_KEY or LLM_API_KEY in your project .env.")

llm_client = OpenAI(
    base_url=os.getenv("LLM_BASE_URL", "https://ollama.com/v1"),
    api_key=api_key,
    timeout=60,
    max_retries=0,
)
agent = AgenticRAG(
    llm_client,
    model=os.getenv("LLM_MODEL", DEFAULT_MODEL),
)

In [12]:
question = "How do I remove duplicates from a Python list while keeping their order?"
display(Markdown(agent.ask(question)))

Depending on your Python version and whether your list items are hashable (like strings or integers), there are several ways to remove duplicates while preserving order.

### 1. For Python 3.7+ (Recommended)
Since Python 3.7, standard dictionaries are guaranteed to maintain insertion order. The most concise and efficient way to remove duplicates is using `dict.fromkeys()`.

```python
items = [1, 2, 1, 3, 2, 1]
unique_items = list(dict.fromkeys(items))
# Output: [1, 2, 3]
```
**Why it works:** `dict.fromkeys()` creates a dictionary where the list elements are keys. Because dictionary keys must be unique, duplicates are removed, and since order is preserved, the original sequence remains [1](https://stackoverflow.com/a/11059378).

### 2. For Older Versions or Explicit Logic
If you are using an older version of Python or prefer a more explicit approach, you can use a `set` to keep track of items you have already seen while iterating through the list [1](https://stackoverflow.com/a/11059378).

```python
seen = set()
items = [1, 2, 1, 3, 2, 1]
unique_items = [x for x in items if not (x in seen or seen.add(x))]
# Output: [1, 2, 3]
```
**How it works:** This list comprehension checks if an item is in the `seen` set. If it is not, `seen.add(x)` is called (which returns `None`, making the expression evaluate to `False` and allowing the item to be kept) [1](https://stackoverflow.com/a/11059378).

### Summary Table

| Method | Time Complexity | Requirement | Note |
| :--- | :--- | :--- | :--- |
| `dict.fromkeys()` | $O(n)$ | Hashable items | Cleanest syntax for Python 3.7+ |
| `set` + List Comp | $O(n)$ | Hashable items | Works across all Python versions |

**Important Caveat:** Both methods above require the items in your list to be **hashable** (e.g., they cannot be lists or dictionaries). If you are dealing with unhashable items, you would need to iterate manually and check for existence using a different method, which would increase the time complexity to $O(n^2)$.

Sources:

1. [Best way to eliminate duplicates in a list, but keep previous relative order](https://stackoverflow.com/a/11059378) — kindall, CC BY-SA 3.0
2. [Best way to eliminate duplicates in a list, but keep previous relative order](https://stackoverflow.com/a/11059418) — JustinDanielson, CC BY-SA 3.0

Edit `question` and rerun the last cell to ask something else. When finished, call `llm_client.close()` or shut down the kernel.
